# Ch 9. 현대 블록 — RoPE · RMSNorm · SwiGLU · GQA

2017년 원조 트랜스포머에서 2024년 표준으로 교체된 4 블록의 이유와 코드.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/desty/study-tiny-llm/blob/main/notebooks/part3/ch09_modern_blocks.ipynb)

In [ ]:
# 필요한 경우 설치
# !pip install torch

import torch
import torch.nn as nn
import torch.nn.functional as F

device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'device: {device}')
print(f'torch version: {torch.__version__}')

## 최소 예제 1 — RoPE (회전 위치 인코딩)

PE를 더하는 대신 **Q, K를 회전**시킨다.
dot product는 두 토큰의 **상대 거리만**이 결정된다.

- attention 직전에 Q, K 만 회전. V 는 그대로.
- 짝/홀 차원을 2D 회전 행렬로 묶어 회전.

In [ ]:
import torch

def precompute_rope(dim, max_len, base=10000.0):
    """미리 cos/sin 테이블."""
    inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))   # (dim/2,)
    t = torch.arange(max_len).float()
    freqs = t[:, None] * inv_freq[None, :]                                # (T, dim/2)
    return freqs.cos(), freqs.sin()                                       # 둘 다 (T, dim/2)

def apply_rope(x, cos, sin):                                              # (1)
    """x: (B, H, T, head_dim). cos/sin: (T, head_dim/2)."""
    x1, x2 = x[..., 0::2], x[..., 1::2]                                   # 짝/홀 분리
    rotated_1 = x1 * cos - x2 * sin
    rotated_2 = x1 * sin + x2 * cos
    return torch.stack([rotated_1, rotated_2], dim=-1).flatten(-2)        # (2)

# 사용 예시
B, H, T, head_dim = 1, 4, 8, 16
max_len = 512

Q = torch.randn(B, H, T, head_dim)
K = torch.randn(B, H, T, head_dim)

cos, sin = precompute_rope(head_dim, max_len)
Q_rot = apply_rope(Q, cos[:T], sin[:T])                                   # K 도 똑같이
K_rot = apply_rope(K, cos[:T], sin[:T])
# V 는 회전 안 함 (원 RoPE 정의)

print("Q shape:", Q.shape)
print("Q_rot shape:", Q_rot.shape)  # 같아야 함
print("cos table shape:", cos.shape)

## 최소 예제 2 — RMSNorm

LayerNorm에서 평균 빼기·shift 더하기를 제거.
RMS (root mean square) 로만 정규화 → 연산 7~10% 절감, 같은 성능.

$$\text{RMSNorm}(x) = \gamma \cdot \frac{x}{\sqrt{\frac{1}{d}\sum x_i^2 + \epsilon}}$$

In [ ]:
import torch
import torch.nn as nn

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        rms = x.pow(2).mean(-1, keepdim=True).sqrt()
        return self.gamma * x / (rms + self.eps)

# 검증
x = torch.randn(2, 8, 256)  # (B, T, D)
norm = RMSNorm(256)
out = norm(x)
print("RMSNorm output shape:", out.shape)
print("gamma shape:", norm.gamma.shape)

## 최소 예제 3 — SwiGLU

$$\text{SwiGLU}(x) = W_2 \cdot \big(\text{SiLU}(W_1 x) \odot W_3 x\big)$$

투영을 두 개 만들어 하나는 SiLU 통과, 하나는 그대로. **원소별 곱** 으로 게이팅.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SwiGLU(nn.Module):
    def __init__(self, dim, hidden):
        super().__init__()
        # 표준 FFN 은 W1, W2 두 개. SwiGLU 는 W1, W2, W3 세 개         (1)
        self.w1 = nn.Linear(dim, hidden, bias=False)
        self.w2 = nn.Linear(hidden, dim, bias=False)
        self.w3 = nn.Linear(dim, hidden, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))                  # (2)

# 검증
dim = 256
hidden = int(8 * dim / 3)  # 표준: (8/3) * dim
ffn = SwiGLU(dim, hidden)
x = torch.randn(2, 8, dim)  # (B, T, D)
out = ffn(x)
print("SwiGLU output shape:", out.shape)
print(f"dim={dim}, hidden={hidden}")

## 실전 — GQA (Grouped Query Attention)

**Q head는 그대로 (32), K/V head만 줄여서 (e.g. 8) 그룹 공유**.
KV cache가 K/V head 수에 비례 → 32 → 8이면 **메모리 1/4**.

attention 직전 K/V를 head 차원에서 **반복** 시키면 끝.

In [ ]:
import torch
import torch.nn.functional as F

def repeat_kv(x, n_rep):
    """x: (B, T, H_kv, d_h). 출력: (B, T, H_kv * n_rep, d_h)."""
    B, T, H_kv, d_h = x.shape
    if n_rep == 1:
        return x
    return (x[:, :, :, None, :]
              .expand(B, T, H_kv, n_rep, d_h)
              .reshape(B, T, H_kv * n_rep, d_h))

# 예시: Q=8 heads, KV=2 heads
B, T = 1, 16
n_q_heads = 8
n_kv_heads = 2
d_h = 32

Q = torch.randn(B, T, n_q_heads, d_h)
K = torch.randn(B, T, n_kv_heads, d_h)
V = torch.randn(B, T, n_kv_heads, d_h)

# attention 직전
n_rep = n_q_heads // n_kv_heads          # 8 / 2 = 4
K_exp = repeat_kv(K, n_rep)
V_exp = repeat_kv(V, n_rep)

print(f"K before: {K.shape}  →  K after: {K_exp.shape}")
print(f"Q shape: {Q.shape}, K expanded: {K_exp.shape}")

# KV cache 메모리 절감 계산
def kv_cache_mb(L, H_kv, d_h, T, bytes_per=2):
    return 2 * L * H_kv * d_h * T * bytes_per / 1e6

print(f"\nKV cache (MHA, H=8, T=512): {kv_cache_mb(6, 8, 40, 512):.2f} MB")
print(f"KV cache (GQA H_kv=2, T=512): {kv_cache_mb(6, 2, 40, 512):.2f} MB")

## 실전 — RMSNorm vs LayerNorm 속도 비교

In [ ]:
import torch
import torch.nn as nn
import time

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(dim))
        self.eps = eps
    def forward(self, x):
        rms = x.pow(2).mean(-1, keepdim=True).sqrt()
        return self.gamma * x / (rms + self.eps)

B, T, D = 8, 512, 512
x = torch.randn(B, T, D)

rms_norm = RMSNorm(D)
layer_norm = nn.LayerNorm(D)

N = 100

# RMSNorm 시간
start = time.perf_counter()
for _ in range(N):
    _ = rms_norm(x)
rms_time = (time.perf_counter() - start) / N * 1000

# LayerNorm 시간
start = time.perf_counter()
for _ in range(N):
    _ = layer_norm(x)
ln_time = (time.perf_counter() - start) / N * 1000

print(f"RMSNorm: {rms_time:.3f} ms")
print(f"LayerNorm: {ln_time:.3f} ms")
print(f"RMSNorm speedup: {ln_time / rms_time:.2f}x")

## 연습문제

1. RMSNorm 과 LayerNorm 의 forward 시간을 같은 입력 (B=8, T=512, D=512) 으로 100 회 측정해 평균 비교. 7~10% 절감 신호 보이는가?
2. RoPE 의 `base=10000` 을 `base=100000` 으로 바꿔 같은 길이 학습 후 외삽 (4K → 16K) 평가. 어느 쪽이 더 안정한가?
3. SwiGLU 의 `hidden = 4 × dim` vs `hidden = (8/3) × dim` 두 설정으로 작은 학습 (10M, 50M 토큰) 을 돌려 손실 곡선 비교.
4. GQA-8 (Q=32, KV=8) 와 MHA (Q=32, KV=32) 의 KV cache 메모리를 seq=2K, layer=12, head_dim=64, fp16 기준으로 직접 계산하라.
5. **(생각해볼 것)** 2017 년에 누군가 "이 4가지를 동시에 바꾸자" 고 제안했다면 받아들여졌을까? 왜 한 번에 하나씩 도입됐는지, 분야 진화 관점에서 한 단락.

In [ ]:
# 연습문제 1: RMSNorm vs LayerNorm 100회 평균 측정 (위 셀 참고, 정밀 측정)


In [ ]:
# 연습문제 2: RoPE base=10000 vs base=100000 비교


In [ ]:
# 연습문제 3: SwiGLU hidden=4*dim vs hidden=(8/3)*dim 파라미터 수 비교
dim = 256
hidden_4x = 4 * dim
hidden_83x = int(8 * dim / 3)

# 파라미터 수 계산 (W1, W2, W3)
params_4x = dim * hidden_4x + hidden_4x * dim + dim * hidden_4x
params_83x = dim * hidden_83x + hidden_83x * dim + dim * hidden_83x

print(f"hidden=4x ({hidden_4x}): {params_4x/1e6:.3f}M params")
print(f"hidden=(8/3)x ({hidden_83x}): {params_83x/1e6:.3f}M params")
print(f"비율: {params_4x / params_83x:.2f}x")

In [ ]:
# 연습문제 4: GQA-8 vs MHA KV cache 메모리 계산
def kv_cache_gb(L, H_kv, d_h, T, bytes_per=2):
    return 2 * L * H_kv * d_h * T * bytes_per / 1e9

# seq=2K, layer=12, head_dim=64, fp16
L, d_h, T = 12, 64, 2048
print(f"MHA  (KV=32): {kv_cache_gb(L, 32, d_h, T):.3f} GB")
print(f"GQA-8 (KV=8): {kv_cache_gb(L, 8, d_h, T):.3f} GB")
print(f"절감: {kv_cache_gb(L, 32, d_h, T) / kv_cache_gb(L, 8, d_h, T):.1f}x")

In [ ]:
# 연습문제 5: 4가지 블록이 한 번에 도입되지 않은 이유 — 메모나 생각 정리
